In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")


Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.bronze (
    kafka_key STRING,
    raw_value STRING,
    topic STRING,
    partition INT,
    offset BIGINT,
    kafka_timestamp TIMESTAMP
) USING iceberg
""")

DataFrame[]

In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [4]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [5]:
# SEMINAR TASK 
# Consume a few messages from the topic using kafka-console-consumer.sh to verify they are there

# docker exec kafka sh -c "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic taxi-trips --from-beginning  --max-messages 5"


In [ ]:
# starts the query
bronze = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("kafka_key"),
        F.col("value").cast("string").alias("raw_value"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.col("timestamp").alias("kafka_timestamp")
    )
)

query = (
    bronze.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/taxi_bronze")
    .toTable("lakehouse.taxi.bronze")
)

query.awaitTermination()

In [7]:
# RUN THIS TO STOP THE QUERY
query.stop()

In [8]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 10").show()

+--------+
|count(1)|
+--------+
|     460|
+--------+

+---------+--------------------+----------+---------+------+--------------------+
|kafka_key|           raw_value|     topic|partition|offset|     kafka_timestamp|
+---------+--------------------+----------+---------+------+--------------------+
|        1|{"VendorID": 1, "...|taxi-trips|        0|     0|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     1|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     2|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     3|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     4|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     5|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     6|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips|        0|     7|2026-04-03 11:13:...|
|        1|{"VendorID": 1, "...|taxi-trips

In [ ]:
# SILVER

In [ ]:
# GOLD pole obvs neid veel testinud, lihtsalt panen esialgse lahenduse üles, et pärast ei peaks mergega dealima

# VARIANT A: streaming 

# spark.sql("""
# CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
#     day DATE,
#     pickup_zone STRING,
#     trip_count LONG,
#     avg_distance DOUBLE,
#     avg_fare DOUBLE,
#     avg_total DOUBLE,
#     tip_rate_pct DOUBLE,
#     total_revenue DOUBLE
#     ) USING iceberg
# PARTITIONED BY (day)
# """)

# gold_source = (
#     spark.readStream
#     .format("iceberg")
#     .option("stream-from-timestamp", "0")
#     .load("lakehouse.taxi.silver")
# )

# gold = (
#     gold_source
#     .withColumn("day", F.to_date("tpep_pickup_datetime"))
#     .groupBy("day", "pickup_zone")
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
#         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
#         F.round(F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0))
#                 / F.count("*") * 100, 2).alias("tip_rate_pct"),
#         F.round(F.avg("total_amount"), 2).alias("avg_total"),
#         F.round(F.sum("total_amount"), 2).alias("total_revenue"),
#     )
# )

#def upsert_gold(batch_df, batch_id):
#    batch_df.createOrReplaceTempView("gold_batch")
#    spark.sql("""
#        MERGE INTO lakehouse.taxi.gold AS target
#        USING gold_batch AS source
#        ON target.day = source.day AND target.pickup_zone = source.pickup_zone
#        WHEN MATCHED THEN UPDATE SET *
#        WHEN NOT MATCHED THEN INSERT *
#    """)
    
#query_gold = (
#    gold_stream.writeStream
#    .outputMode("update")     
#    .trigger(processingTime="1 minute")
#    .option("checkpointLocation", "s3://checkpoints/gold") see vaja ülev vaadata!! 
#    .foreachBatch(upsert_gold)
#    .start()
#)

# query_gold.awaitTermination()


In [ ]:
# GOLD

# VARIANT B: BATCH 

# spark.sql("""
# CREATE TABLE IF NOT EXISTS lakehouse.taxi.gold (
#     day DATE,
#     pickup_zone STRING,
#     trip_count LONG,
#     avg_distance DOUBLE,
#     avg_fare DOUBLE,
#     avg_total DOUBLE,
#     tip_rate_pct DOUBLE,
#     total_revenue DOUBLE
#     ) USING iceberg
# PARTITIONED BY (day)
# """)

# gold_df = (
#     spark.table("lakehouse.taxi.silver")
#     .withColumn("day", F.to_date("tpep_pickup_datetime"))
#     # hourly: .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime"))
#     .groupBy("day", "pickup_zone")
#     # hourly: .groupBy("hour", "pickup_zone")
#     .agg(
#         F.count("*").alias("trip_count"),
#         F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
#         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
#         F.round(F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0)).cast(DoubleType())
#                 / F.count("*") * 100, 2).alias("tip_rate_pct"),
#         F.round(F.avg("total_amount"), 2).alias("avg_total"),
#         F.round(F.sum("total_amount"), 2).alias("total_revenue"),
#     )
# )


# todo mõelda, kas mõistlikum teha hourly või daily, kui hour, siis hour TIMESTAMP. kui teha tunni kaupa, siis .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime")) ja .groupBy("hour", "pickup_zone")
# gold_df.writeTo("lakehouse.taxi.gold").append()
